# 03 ESOL Dataset Curation: Labels, Duplicates, and Scaffolds

This section treats ESOL as a real dataset to inspect, not as a CSV to send directly into a model. The key questions are: what does the `logS` label mean, where does the data come from, how can derived features remain traceable, and what model claim can each split support?

## Intuition

The first step in AI4Chem is not modeling, but data. Ask:

- What are the label units?
- Which paper or database is the data from?
- Can the SMILES be parsed?
- Are there duplicate or near-duplicate structures?
- Is the test set too similar to the training set?

ESOL is a classic small aqueous-solubility dataset often used for teaching molecular property prediction. It is useful for practicing descriptors, fingerprints, baselines, random split, and scaffold split. It is not a modern deployment-grade solubility benchmark.

## Mathematical and Chemical Definitions

The ESOL label is `log solubility (mol/L)`, written here as `logS`:

```text
S: aqueous solubility in mol/L
logS = log10(S / (1 mol/L))
```

For example, `logS = -3` means a solubility of about `10^-3 mol/L`. Many organic small molecules have solubility below 1 mol/L, so logS is often negative.

The ESOL data used here comes from Delaney's aqueous-solubility dataset:

> Delaney, J. S. ESOL: estimating aqueous solubility directly from molecular structure.
> Journal of Chemical Information and Computer Sciences, 2004. DOI: 10.1021/ci034243x

A scaffold is an approximation of the molecular core. A scaffold split tries to put different core structures into train and test sets, giving a stricter estimate of extrapolation to new cores.

## Split Hierarchy: Splits Define the Claim

A data split is not a minor technical detail. It defines what the model is being tested on:

| split | How it works | Claim it supports | What it cannot show |
| --- | --- | --- | --- |
| random split | Randomly sample a test set | Interpolation on similar molecules from the same distribution | Generalization to new scaffolds |
| scaffold split | Group by Murcko scaffold and keep test scaffolds mostly unseen in training | Initial extrapolation to new core structures | Still may have nearest-neighbor leakage; not source/time extrapolation |
| cluster split | Cluster by fingerprint similarity, then split by cluster | More direct control of structural similarity | Threshold choice affects the conclusion |
| source split | Split by experiment source, lab, paper, or assay | Robustness across protocols or sources | ESOL CSV lacks enough source metadata |
| time split | Split by publication year or database-entry time | Closer to future deployment | Needs reliable timestamps; not available here |
| external test | Test on a fully independent dataset | Closest to real generalization check | Expensive; label definitions must match |

In short, random split supports "interpolation among similar data". Claims about new scaffolds, new sources, or future data require stronger splits or external tests. This section inspects ESOL structure and scaffold distribution; `04` builds random-split models, and `05` compares random split with scaffold split.

## Prepare Paths

This cell only finds the `materials/` root and `data/` paths. The same paths work whether the notebook is opened from `materials/` or from `notebooks/`.

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

## Prepare RDKit Helper Functions

This cell defines reusable functions for parsing SMILES, generating canonical SMILES, computing descriptors, extracting Murcko scaffolds, and converting raw ESOL rows into a derived modeling table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

sns.set_theme(style="whitegrid", context="notebook")


def mol_from_smiles(smiles):
    if pd.isna(smiles):
        return None
    return Chem.MolFromSmiles(str(smiles).strip())


def canonical_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def scaffold_smiles(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def descriptor_record(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "canonical_smiles": canonical_smiles(smiles),
        "scaffold": scaffold_smiles(mol),
    }


def build_esol_features():
    raw = pd.read_csv(RAW / "esol.csv")
    rows = []
    for row_id, row in raw.reset_index(drop=True).iterrows():
        desc = descriptor_record(row["smiles"])
        if desc is None:
            continue
        desc.update(
            {
                "row_id": row_id,
                "smiles": str(row["smiles"]).strip(),
                "logS": float(row["log solubility (mol/L)"]),
            }
        )
        rows.append(desc)
    return pd.DataFrame(rows)


def fingerprint_array(smiles, n_bits=1024):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    matrix = np.zeros((len(smiles), n_bits), dtype=np.int8)
    for idx, smi in enumerate(smiles):
        fp = generator.GetFingerprint(mol_from_smiles(smi))
        DataStructs.ConvertToNumpyArray(fp, matrix[idx])
    return matrix


DESCRIPTOR_COLUMNS = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "AromaticRings",
    "FractionCSP3",
    "HeavyAtomCount",
]

## Read Raw ESOL

First inspect the raw CSV: row count, column names, and a few examples. No overwriting happens here, so raw data provenance remains intact.

In [ ]:
# The raw table only contains SMILES and experimental labels; derived columns are created later and not written back to the raw CSV.
raw = pd.read_csv(RAW / "esol.csv")
print(raw.shape)
raw.head()

## Create a Traceable Derived Table

The next cell creates a derived table: it keeps the raw row id and adds canonical SMILES, descriptors, and scaffold. It does not overwrite the raw CSV.

In [ ]:
# build_esol_features skips SMILES that RDKit cannot parse and keeps the original row_id.
esol = build_esol_features()

print("raw rows:", len(raw))
print("valid molecule rows:", len(esol))
print("invalid rows:", len(raw) - len(esol))
esol[["row_id", "smiles", "canonical_smiles", "scaffold", "logS"]].head()

## Check Duplicate Structures

The same molecule can have multiple SMILES forms. Grouping by canonical SMILES can reveal identity-leakage risk: if the same structure appears in both train and test, test scores may be inflated.

In [ ]:
# Identical canonical_smiles means RDKit treats them as the same standardized molecular graph.
duplicate_counts = esol["canonical_smiles"].value_counts()
num_duplicate_structures = int((duplicate_counts > 1).sum())
print("duplicated canonical structures:", num_duplicate_structures)

if num_duplicate_structures:
    duplicated = esol[esol["canonical_smiles"].isin(duplicate_counts[duplicate_counts > 1].index)]
    display(duplicated.sort_values("canonical_smiles").head(10))
else:
    print("No duplicate canonical SMILES found.")

## Inspect Label Distribution and a Simple Trend

These plots answer two basic questions: what is the approximate range of `logS`, and is there a rough relationship between molecular weight and solubility? This is not a modeling conclusion; it is a way to understand labels and obvious outliers.

In [ ]:
# The left plot shows the label distribution; the right plot checks one simple descriptor-label relationship.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(esol["logS"], bins=35, ax=axes[0])
axes[0].set_title("ESOL label distribution")
axes[0].set_xlabel("logS")

sns.scatterplot(data=esol, x="MolWt", y="logS", alpha=0.6, ax=axes[1])
axes[1].set_title("Molecular weight vs logS")
plt.tight_layout()

## Inspect Scaffold Distribution

Scaffold distribution affects later splits. If a scaffold is very frequent, random split will likely place similar cores in both train and test. Scaffold split is closer to the question "can the model handle new core structures?"

In [ ]:
# Empty scaffolds often occur for acyclic molecules; here they are displayed with a readable label.
scaffold_summary = (
    esol["scaffold"]
    .replace("", "[no ring scaffold]")
    .value_counts()
    .head(10)
    .rename_axis("scaffold")
    .reset_index(name="count")
)
scaffold_summary

## Observation Questions

1. What range does ESOL logS roughly cover?
2. Is there an obvious trend between molecular weight and solubility? What are the exceptions?
3. Why should a data card record provenance, units, and limitations?

### Hints

1. Check the histogram range. Most molecules have negative logS, meaning molar solubility below 1 mol/L.
2. Higher molecular weight often means lower solubility, but this is only a weak trend. Molecules with many hydrogen-bond donors/acceptors, strongly polar groups, or ionizable states may deviate.
3. Provenance, units, and limitations determine whether labels are comparable. Without a data card, it is hard to tell whether a model learned solubility chemistry, source bias, or accidental data-processing patterns.

## Summary

Dataset curation is not merely "cleaning dirty data". It builds a traceable evidence chain. Keep raw data, make derived features reproducible, and remember that duplicates and scaffolds affect later model evaluation.